# GPU Memory Hierarchy

Master the GPU memory hierarchy: global, shared, register, and constant memory. Learn to use shared memory for inter-thread communication, synchronize with syncthreads, and optimize memory access patterns for coalescing.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-memory-model)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: Memory Types

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda
from numba import float32

# ============================================
# Demonstrating Memory Hierarchy Performance
# ============================================

# Kernel 1: Repeated reads from GLOBAL memory
@cuda.jit
def global_memory_kernel(data, output):
    idx = cuda.grid(1)
    if idx < data.size:
        # Read the same value from global memory 10 times
        total = 0.0
        for i in range(10):
            total += data[idx]  # Each read goes to global memory
        output[idx] = total

# Kernel 2: Load to register once, reuse from register
@cuda.jit
def register_kernel(data, output):
    idx = cuda.grid(1)
    if idx < data.size:
        val = data[idx]  # One global read -> register
        total = 0.0
        for i in range(10):
            total += val  # Reads from register (essentially free)
        output[idx] = total

# Kernel 3: Using shared memory for inter-thread data sharing
@cuda.jit
def shared_memory_kernel(data, output):
    sdata = cuda.shared.array(256, dtype=float32)
    idx = cuda.grid(1)
    tid = cuda.threadIdx.x

    # Load into shared memory
    if idx < data.size:
        sdata[tid] = data[idx]
    cuda.syncthreads()

    # Each thread reads its neighbor's value from shared memory
    if idx < data.size and tid > 0:
        output[idx] = sdata[tid] + sdata[tid - 1]  # Fast: shared memory
    elif idx < data.size:
        output[idx] = sdata[tid]

# Setup
n = 10_000_000
data = np.random.randn(n).astype(np.float32)
d_data = cuda.to_device(data)
d_out = cuda.device_array(n, dtype=np.float32)

threads = 256
blocks = (n + threads - 1) // threads

# Warm up all kernels
global_memory_kernel[blocks, threads](d_data, d_out)
register_kernel[blocks, threads](d_data, d_out)
shared_memory_kernel[blocks, threads](d_data, d_out)
cuda.synchronize()

# Benchmark
iterations = 100

start = time.perf_counter()
for _ in range(iterations):
    global_memory_kernel[blocks, threads](d_data, d_out)
cuda.synchronize()
global_time = (time.perf_counter() - start) / iterations

start = time.perf_counter()
for _ in range(iterations):
    register_kernel[blocks, threads](d_data, d_out)
cuda.synchronize()
register_time = (time.perf_counter() - start) / iterations

start = time.perf_counter()
for _ in range(iterations):
    shared_memory_kernel[blocks, threads](d_data, d_out)
cuda.synchronize()
shared_time = (time.perf_counter() - start) / iterations

print("=== Memory Hierarchy Performance ===")
print(f"Global memory (10 reads): {global_time*1000:.3f} ms")
print(f"Register (1 read, 10 reuse): {register_time*1000:.3f} ms")
print(f"Shared memory (neighbor access): {shared_time*1000:.3f} ms")
print(f"\nSpeedup (register vs global): {global_time/register_time:.2f}x")
print(f"Key insight: Loading to a register once and reusing is much")
print(f"faster than reading global memory repeatedly.")

# ============================================
# Query memory information
# ============================================
device = cuda.get_current_device()
print(f"\n=== GPU Memory Info ===")
print(f"Max shared memory per block: {device.MAX_SHARED_MEMORY_PER_BLOCK:,} bytes")
print(f"  = {device.MAX_SHARED_MEMORY_PER_BLOCK / 1024:.0f} KB")
print(f"Multiprocessor count: {device.MULTIPROCESSOR_COUNT}")
print(f"Warp size: {device.WARP_SIZE}")

# Show GPU memory usage
free, total = cuda.current_context().get_memory_info()
print(f"\nGPU VRAM: {total / 1e9:.1f} GB total, {free / 1e9:.1f} GB free")
print(f"  Used: {(total - free) / 1e6:.0f} MB")

### Lesson example: Using Shared Memory

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda, float32

# ============================================
# Shared Memory: Parallel Reduction (Block Sum)
# ============================================

BLOCK_SIZE = 256

@cuda.jit
def block_reduce_sum(data, partial_sums):
    """Each block computes the sum of its elements using shared memory."""
    shared = cuda.shared.array(BLOCK_SIZE, dtype=float32)
    tid = cuda.threadIdx.x
    idx = cuda.grid(1)

    # Load into shared memory (0 for out-of-bounds threads)
    shared[tid] = data[idx] if idx < data.size else 0.0
    cuda.syncthreads()

    # Tree reduction
    stride = BLOCK_SIZE // 2
    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]
        cuda.syncthreads()
        stride //= 2

    # Thread 0 writes the block sum
    if tid == 0:
        partial_sums[cuda.blockIdx.x] = shared[0]

def gpu_sum(data):
    """Sum a large array using two-pass reduction."""
    n = data.size
    d_data = cuda.to_device(data)

    # Pass 1: reduce each block
    blocks = (n + BLOCK_SIZE - 1) // BLOCK_SIZE
    d_partial = cuda.device_array(blocks, dtype=np.float32)
    block_reduce_sum[blocks, BLOCK_SIZE](d_data, d_partial)

    # Pass 2: reduce the partial sums (if they fit in one block)
    if blocks <= BLOCK_SIZE:
        d_result = cuda.device_array(1, dtype=np.float32)
        block_reduce_sum[1, BLOCK_SIZE](d_partial, d_result)
        return d_result.copy_to_host()[0]
    else:
        # Recursively reduce
        return gpu_sum(d_partial.copy_to_host())

# Test reduction
print("=== Parallel Reduction (Sum) ===")
n = 1_000_000
data = np.random.randn(n).astype(np.float32)

gpu_result = gpu_sum(data)
cpu_result = np.sum(data)
print(f"GPU sum: {gpu_result:.4f}")
print(f"CPU sum: {cpu_result:.4f}")
print(f"Difference: {abs(gpu_result - cpu_result):.6f}")
print(f"(Small difference is expected due to float32 accumulation order)")

# ============================================
# Shared Memory: Neighbor Averaging (Stencil)
# ============================================
print("\n=== Stencil Operation (3-point average) ===")

# WITHOUT shared memory: each element reads 3 values from global memory
@cuda.jit
def stencil_global(data, output):
    idx = cuda.grid(1)
    if idx > 0 and idx < data.size - 1:
        output[idx] = (data[idx-1] + data[idx] + data[idx+1]) / 3.0

# WITH shared memory: load block into shared, read neighbors from there
@cuda.jit
def stencil_shared(data, output):
    shared = cuda.shared.array(BLOCK_SIZE + 2, dtype=float32)  # +2 for halo elements
    tid = cuda.threadIdx.x
    idx = cuda.grid(1)

    # Load main data
    if idx < data.size:
        shared[tid + 1] = data[idx]  # +1 offset for left halo

    # Load halo elements (boundary of the block)
    if tid == 0 and idx > 0:
        shared[0] = data[idx - 1]  # Left halo
    if tid == BLOCK_SIZE - 1 and idx < data.size - 1:
        shared[BLOCK_SIZE + 1] = data[idx + 1]  # Right halo

    cuda.syncthreads()

    # Compute stencil from shared memory
    if idx > 0 and idx < data.size - 1:
        output[idx] = (shared[tid] + shared[tid + 1] + shared[tid + 2]) / 3.0

n = 10_000_000
data = np.random.randn(n).astype(np.float32)
d_data = cuda.to_device(data)
d_out_global = cuda.device_array(n, dtype=np.float32)
d_out_shared = cuda.device_array(n, dtype=np.float32)

blocks = (n + BLOCK_SIZE - 1) // BLOCK_SIZE

# Warm up
stencil_global[blocks, BLOCK_SIZE](d_data, d_out_global)
stencil_shared[blocks, BLOCK_SIZE](d_data, d_out_shared)
cuda.synchronize()

# Benchmark
iters = 100
start = time.perf_counter()
for _ in range(iters):
    stencil_global[blocks, BLOCK_SIZE](d_data, d_out_global)
cuda.synchronize()
global_time = (time.perf_counter() - start) / iters

start = time.perf_counter()
for _ in range(iters):
    stencil_shared[blocks, BLOCK_SIZE](d_data, d_out_shared)
cuda.synchronize()
shared_time = (time.perf_counter() - start) / iters

# Verify correctness
out_g = d_out_global.copy_to_host()
out_s = d_out_shared.copy_to_host()
assert np.allclose(out_g[1:-1], out_s[1:-1], atol=1e-5)

print(f"Global memory version: {global_time*1000:.3f} ms")
print(f"Shared memory version: {shared_time*1000:.3f} ms")
print(f"Speedup: {global_time/shared_time:.2f}x")
print(f"\nFor stencil ops, shared memory helps because each element")
print(f"is read by 3 threads. Loading to shared memory turns 3 global")
print(f"reads into 1 global read + 3 fast shared reads.")

### Lesson example: Memory Coalescing

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda, float32

# ============================================
# Memory Coalescing: Stride Access Benchmark
# ============================================

@cuda.jit
def stride_access(data, output, stride):
    """Access data with a given stride between threads."""
    idx = cuda.grid(1)
    data_idx = idx * stride
    if data_idx < data.size:
        output[idx] = data[data_idx]

@cuda.jit
def coalesced_access(data, output):
    """Access consecutive elements (stride=1, coalesced)."""
    idx = cuda.grid(1)
    if idx < data.size:
        output[idx] = data[idx]

print("=== Coalescing: Stride vs Consecutive Access ===")
n = 10_000_000
data = np.random.randn(n).astype(np.float32)
d_data = cuda.to_device(data)

threads = 256
blocks = (n // 32 + threads - 1) // threads  # enough for stride-32

d_out = cuda.device_array(n, dtype=np.float32)

# Warm up
coalesced_access[blocks, threads](d_data, d_out)
cuda.synchronize()

iters = 100

for stride in [1, 2, 4, 8, 16, 32]:
    n_elements = n // stride
    bl = (n_elements + threads - 1) // threads
    d_out_s = cuda.device_array(n_elements, dtype=np.float32)

    # Warm up
    stride_access[bl, threads](d_data, d_out_s, stride)
    cuda.synchronize()

    start = time.perf_counter()
    for _ in range(iters):
        stride_access[bl, threads](d_data, d_out_s, stride)
    cuda.synchronize()
    t = (time.perf_counter() - start) / iters

    bytes_moved = n_elements * 4 * 2  # read + write
    bw = bytes_moved / t / 1e9
    print(f"  Stride {stride:>2}: {t*1000:.3f} ms, effective BW: {bw:.1f} GB/s")

# ============================================
# AoS vs SoA Demonstration
# ============================================
print("\n=== AoS vs SoA Layout ===")

N_PARTICLES = 2_000_000

# AoS: each particle = [x, y, z, vx, vy, vz]
aos = np.random.randn(N_PARTICLES, 6).astype(np.float32)
d_aos = cuda.to_device(aos)

# SoA: separate arrays for each property
x = np.random.randn(N_PARTICLES).astype(np.float32)
y = np.random.randn(N_PARTICLES).astype(np.float32)
z = np.random.randn(N_PARTICLES).astype(np.float32)
vx = np.random.randn(N_PARTICLES).astype(np.float32)
vy = np.random.randn(N_PARTICLES).astype(np.float32)
vz = np.random.randn(N_PARTICLES).astype(np.float32)
d_x, d_y, d_z = cuda.to_device(x), cuda.to_device(y), cuda.to_device(z)
d_vx, d_vy, d_vz = cuda.to_device(vx), cuda.to_device(vy), cuda.to_device(vz)

@cuda.jit
def update_aos(particles):
    idx = cuda.grid(1)
    if idx < particles.shape[0]:
        particles[idx, 0] += particles[idx, 3]  # x += vx (stride-6 access!)
        particles[idx, 1] += particles[idx, 4]  # y += vy
        particles[idx, 2] += particles[idx, 5]  # z += vz

@cuda.jit
def update_soa(x, y, z, vx, vy, vz):
    idx = cuda.grid(1)
    if idx < x.size:
        x[idx] += vx[idx]  # Coalesced!
        y[idx] += vy[idx]
        z[idx] += vz[idx]

blocks_p = (N_PARTICLES + threads - 1) // threads

# Warm up
update_aos[blocks_p, threads](d_aos)
update_soa[blocks_p, threads](d_x, d_y, d_z, d_vx, d_vy, d_vz)
cuda.synchronize()

# Benchmark AoS
start = time.perf_counter()
for _ in range(iters):
    update_aos[blocks_p, threads](d_aos)
cuda.synchronize()
aos_time = (time.perf_counter() - start) / iters

# Benchmark SoA
start = time.perf_counter()
for _ in range(iters):
    update_soa[blocks_p, threads](d_x, d_y, d_z, d_vx, d_vy, d_vz)
cuda.synchronize()
soa_time = (time.perf_counter() - start) / iters

print(f"  AoS (stride-6 access): {aos_time*1000:.3f} ms")
print(f"  SoA (coalesced access): {soa_time*1000:.3f} ms")
print(f"  SoA speedup: {aos_time/soa_time:.2f}x")
print(f"\n  SoA is faster because consecutive threads access")
print(f"  consecutive memory addresses (coalesced).")
print(f"  AoS forces stride-6 access, wasting memory bandwidth.")

# ============================================
# Matrix Transpose: Naive vs Tiled
# ============================================
print("\n=== Matrix Transpose: Naive vs Tiled ===")

TILE_DIM = 32

@cuda.jit
def naive_transpose(A, B):
    col, row = cuda.grid(2)
    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]

@cuda.jit
def tiled_transpose(A, B):
    tile = cuda.shared.array((TILE_DIM, TILE_DIM + 1), dtype=float32)  # +1 padding
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    # Input coordinates
    row = cuda.blockIdx.y * TILE_DIM + ty
    col = cuda.blockIdx.x * TILE_DIM + tx

    # Coalesced read from A
    if row < A.shape[0] and col < A.shape[1]:
        tile[ty, tx] = A[row, col]

    cuda.syncthreads()

    # Output coordinates (swapped blocks)
    new_row = cuda.blockIdx.x * TILE_DIM + ty
    new_col = cuda.blockIdx.y * TILE_DIM + tx

    # Coalesced write to B
    if new_row < B.shape[0] and new_col < B.shape[1]:
        B[new_row, new_col] = tile[tx, ty]  # tx,ty swapped = transposed

M = 4096
A = np.random.randn(M, M).astype(np.float32)
d_A = cuda.to_device(A)
d_B_naive = cuda.device_array((M, M), dtype=np.float32)
d_B_tiled = cuda.device_array((M, M), dtype=np.float32)

# Note: (32, 32) = 1024 threads per block, which is the hardware maximum.
# This works for simple kernels but may fail if the kernel uses many registers.
# An alternative is (32, 8) with each thread processing multiple rows.
TPB_2d = (TILE_DIM, TILE_DIM)
BPG_2d = ((M + TILE_DIM - 1) // TILE_DIM, (M + TILE_DIM - 1) // TILE_DIM)

# Warm up
naive_transpose[BPG_2d, TPB_2d](d_A, d_B_naive)
tiled_transpose[BPG_2d, TPB_2d](d_A, d_B_tiled)
cuda.synchronize()

# Benchmark
start = time.perf_counter()
for _ in range(iters):
    naive_transpose[BPG_2d, TPB_2d](d_A, d_B_naive)
cuda.synchronize()
naive_time = (time.perf_counter() - start) / iters

start = time.perf_counter()
for _ in range(iters):
    tiled_transpose[BPG_2d, TPB_2d](d_A, d_B_tiled)
cuda.synchronize()
tiled_time = (time.perf_counter() - start) / iters

# Verify
assert np.allclose(d_B_naive.copy_to_host(), A.T, atol=1e-5)
assert np.allclose(d_B_tiled.copy_to_host(), A.T, atol=1e-5)

bytes_total = 2 * M * M * 4  # read + write
print(f"  Naive:  {naive_time*1000:.3f} ms ({bytes_total/naive_time/1e9:.0f} GB/s)")
print(f"  Tiled:  {tiled_time*1000:.3f} ms ({bytes_total/tiled_time/1e9:.0f} GB/s)")
print(f"  Speedup: {naive_time/tiled_time:.2f}x")

---

## Exercise: Shared Memory Matrix Transpose


In [ ]:
import numpy as np
import time
from numba import cuda, float32

TILE_DIM = 32

# TODO: Implement naive transpose
@cuda.jit
def naive_transpose(A, B):
    """Transpose A into B without shared memory."""
    pass

# TODO: Implement shared memory transpose
@cuda.jit
def shared_transpose(A, B):
    """Transpose A into B using shared memory for coalesced access."""
    # Hint: allocate shared array with +1 padding
    # tile = cuda.shared.array((TILE_DIM, TILE_DIM + 1), dtype=float32)
    pass

# Test with non-square matrix
M, N = 2048, 4096  # M rows, N columns
A = np.random.randn(M, N).astype(np.float32)

d_A = cuda.to_device(A)
d_B_naive = cuda.device_array((N, M), dtype=np.float32)  # Transposed shape
d_B_shared = cuda.device_array((N, M), dtype=np.float32)

# TODO: Calculate launch configuration
# TODO: Launch both kernels
# TODO: Verify correctness against A.T
# TODO: Benchmark and compute effective bandwidth
# TODO: Calculate % of peak bandwidth (T4 peak: ~320 GB/s)

## Your tasks

Implement a matrix transpose kernel using shared memory to achieve coalesced memory access, and compare it against a naive implementation.

**Requirements:**
1. Implement `naive_transpose(A, B)` that directly reads A[row, col] and writes B[col, row]
2. Implement `shared_transpose(A, B)` that:
   - Loads a TILE_DIM x TILE_DIM tile from A into shared memory (coalesced read)
   - Uses cuda.syncthreads() after the load
   - Writes from shared memory to B with transposed indices (coalesced write)
   - Uses +1 padding on shared memory to avoid bank conflicts
3. Test with a non-square matrix (e.g., 2048 x 4096) to catch row/col bugs
4. Verify both produce the correct transpose (compare with A.T)
5. Benchmark both versions and compute effective bandwidth
6. Calculate what percentage of the GPU's peak bandwidth each achieves

**Hints:**
- TILE_DIM = 32 is standard (matches warp size)
- Shared memory declaration: `cuda.shared.array((TILE_DIM, TILE_DIM + 1), dtype=float32)`
- The +1 eliminates bank conflicts when reading columns
- For the tiled version, swap blockIdx.x and blockIdx.y when computing output coordinates
- Effective bandwidth = 2 * M * N * 4 bytes / time (read + write)
- T4 peak bandwidth is ~320 GB/s
- Note: Using (32, 32) = 1024 threads per block is at the hardware maximum. This works for this simple kernel but may need adjustment for complex kernels that use more registers.

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-memory-model) for the solution and discussion.